# Bank Transaction Default Model

**Author:** Data Science  
**Date:** April 2026  
**Version:** 1.0  

---

## Overview

This notebook builds an **explainable binary classification model** to predict customer default (label = 1)  
from bank transaction data. The pipeline covers:

1. Exploratory Data Analysis (EDA)
2. NLP-based feature extraction from free-text transaction descriptions (`feature_0`)
3. Weight of Evidence / Information Value (WoE/IV) analysis
4. SHAP Recursive Feature Elimination (SHAP RFE)
5. LightGBM with Bayesian Hyperparameter Optimisation
6. Full model evaluation: ROC AUC, KS statistic, SHAP, expected vs actual

**Key Results:** Train AUC = 0.8392 | Test AUC = 0.8252 | Gap = 1.41%  
**Final Features (5):** feature_7, feature_3, feature_6, feature_5, feature_1

## 1. Setup

### 1.1 Install Dependencies


In [3]:
# pip install lightgbm probatus bayesian-optimization shap
# pip install pandas numpy matplotlib seaborn scikit-learn
# Note: optbinning is not compatible with Python 3.13/ARM — custom WoE/IV used instead
import warnings
warnings.filterwarnings('ignore')

### 1.2 Import Libraries


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy.stats import ks_2samp
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    roc_auc_score, roc_curve, confusion_matrix,
    classification_report, precision_recall_curve
)
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
from bayes_opt import BayesianOptimization
from probatus.feature_elimination import ShapRFECV
import shap
import re
import pickle
import os

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120
SEED = 42
np.random.seed(SEED)
print('Libraries loaded successfully')

Libraries loaded successfully


### 1.3 Utility Functions


In [7]:
def check_missing(df):
    """Return missing value summary for all columns."""
    miss = df.isnull().sum()
    miss_pct = (miss / len(df) * 100).round(2)
    result = pd.DataFrame({'Missing': miss, 'Missing_%': miss_pct})
    result = result[result['Missing'] > 0].sort_values('Missing_%', ascending=False)
    return result


def performance_metrics(y_true, y_proba, label=''):
    """Print AUC and KS statistic."""
    auc = roc_auc_score(y_true, y_proba)
    ks  = ks_2samp(y_proba[y_true == 1], y_proba[y_true == 0]).statistic
    print(f'{label:15s}  AUC = {auc:.4f}   KS = {ks:.4f}')
    return auc, ks


def plot_auc2(y_tr, tr_proba, y_te, te_proba, title=''):
    """Side-by-side ROC curve and KS plot for train and test."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # --- ROC ---
    ax = axes[0]
    for y, p, lbl, c in [(y_tr, tr_proba, 'Train', '#1f77b4'),
                          (y_te, te_proba, 'Test',  '#ff7f0e')]:
        fpr, tpr, _ = roc_curve(y, p)
        auc = roc_auc_score(y, p)
        ax.plot(fpr, tpr, color=c, lw=2, label=f'{lbl} (AUC={auc:.4f})')
    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'{title} - ROC Curve')
    ax.legend()

    # --- KS ---
    ax = axes[1]
    for y, p, lbl, c in [(y_tr, tr_proba, 'Train', '#1f77b4'),
                          (y_te, te_proba, 'Test',  '#ff7f0e')]:
        df_ks = pd.DataFrame({'y': y, 'p': p}).sort_values('p', ascending=False).reset_index(drop=True)
        df_ks['cum_events']    = (df_ks['y'] == 1).cumsum() / (df_ks['y'] == 1).sum()
        df_ks['cum_nonevents'] = (df_ks['y'] == 0).cumsum() / (df_ks['y'] == 0).sum()
        ks_val = ks_2samp(p[y == 1], p[y == 0]).statistic
        ax.plot(np.linspace(0, 1, len(df_ks)), df_ks['cum_events'],    color=c, lw=2, linestyle='-',  label=f'{lbl} Events')
        ax.plot(np.linspace(0, 1, len(df_ks)), df_ks['cum_nonevents'], color=c, lw=2, linestyle='--', label=f'{lbl} Non-events (KS={ks_val:.4f})')
    ax.set_xlabel('Population %')
    ax.set_ylabel('Cumulative %')
    ax.set_title(f'{title} - KS Statistic')
    ax.legend(fontsize=8)

    plt.tight_layout()
    plt.show()


def exp_vs_act(y_true, y_proba, n_deciles=10):
    """Decile-level expected vs actual default rate table."""
    df = pd.DataFrame({'y': y_true, 'p': y_proba})
    df = df.sort_values('p', ascending=False).reset_index(drop=True)
    df['decile'] = pd.qcut(df.index, n_deciles, labels=range(1, n_deciles + 1))
    tbl = df.groupby('decile').agg(
        n=('y', 'count'),
        actual_defaults=('y', 'sum'),
        expected_rate=('p', 'mean')
    ).reset_index()
    tbl['actual_rate'] = tbl['actual_defaults'] / tbl['n']
    tbl['actual_rate_pct']   = (tbl['actual_rate']   * 100).round(1)
    tbl['expected_rate_pct'] = (tbl['expected_rate'] * 100).round(1)
    return tbl


print('Utility functions defined')

Utility functions defined


## 2. Data Loading


In [9]:
DATA_PATH = '../data/20230400_Cash_DS.csv'
df_raw = pd.read_csv(DATA_PATH)

print(f'Shape: {df_raw.shape}')
print(f'Columns: {list(df_raw.columns)}')
df_raw.head(3)

Shape: (6000, 16)
Columns: ['feature_0', 'feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5', 'feature_6', 'feature_7', 'feature_8', 'feature_9', 'feature_10', 'feature_11', 'feature_12', 'feature_13', 'feature_14', 'label']


In [10]:
# Target distribution
n_total  = len(df_raw)
n_events = df_raw['label'].sum()
er = n_events / n_total
print(f'Total records : {n_total:,}')
print(f'Defaults (1)  : {int(n_events):,}  ({er:.1%})')
print(f'Non-defaults (0): {n_total - int(n_events):,}  ({1-er:.1%})')
print(f'Class ratio (0:1): {(n_total-int(n_events))/int(n_events):.1f}:1')

Total records : 6,000
Defaults (1)  : 799  (13.3%)
Non-defaults (0): 5,201  (86.7%)
Class ratio (0:1): 6.5:1


## 3. Exploratory Data Analysis


In [12]:
# Missing value analysis
miss = check_missing(df_raw)
print('Missing Value Summary:')
print(miss.to_string())

Missing Value Summary:
            Missing  Missing_%
feature_3      4800      80.00
feature_4      3600      60.00
feature_9       300       5.00
feature_11      180       3.00
feature_12      120       2.00


In [13]:
# Data types and basic stats
print('Data types:')
print(df_raw.dtypes)
print()
print('Numeric summary (selected columns):')
df_raw[['feature_1','feature_5','feature_6','feature_7','label']].describe().round(2)

Data types:
feature_0      object
feature_1     float64
feature_2     float64
feature_3     float64
feature_4     float64
feature_5     float64
feature_6     float64
feature_7     float64
feature_8      object
feature_9     float64
feature_10    float64
feature_11    float64
feature_12    float64
feature_13    float64
feature_14    float64
label           int64
dtype: object


In [14]:
# feature_1 sentinel values
sentinel_count = (df_raw['feature_1'] >= 9999990).sum()
print(f'feature_1 sentinel values (>=9,999,990): {sentinel_count} ({sentinel_count/len(df_raw):.1%})')

# feature_8 encoding check
print(f'\nfeature_8 unique values: {df_raw["feature_8"].unique()}')

# feature_0 sample
print(f'\nfeature_0 sample transactions:')
for t in df_raw['feature_0'].dropna().head(5):
    print(f'  {t}')

feature_1 sentinel values (>=9,999,990): 273 (4.6%)

feature_8 unique values: ['0' '1' 'n']

feature_0 sample transactions:
  RECURRING NETFLIX.COM 12345 LOS ANGELES CA
  AMAZON PRIME*AB1CD2 SEATTLE WA US
  PAYROLL DD EMPLOYER NAME 202304
  ATM CASH WITHDRAWAL BANK BRANCH NYC
  VENMO PAYMENT 1234 TRANSFER


In [15]:
# EDA plots: label distribution + feature distributions
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# 1. Target distribution
ax = axes[0, 0]
counts = df_raw['label'].value_counts()
ax.bar(['Non-Default (0)', 'Default (1)'], counts.values, color=['#2196F3', '#F44336'], width=0.5)
for i, v in enumerate(counts.values):
    ax.text(i, v + 50, f'{v:,}\n({v/len(df_raw):.1%})', ha='center', fontsize=10)
ax.set_title('Target Distribution')
ax.set_ylabel('Count')

# 2. feature_1 distribution
ax = axes[0, 1]
f1_clean = df_raw.loc[df_raw['feature_1'] < 9999990, 'feature_1']
ax.hist(f1_clean, bins=40, color='#2196F3', alpha=0.7, edgecolor='white')
ax.set_title('feature_1 (excl. sentinels)')
ax.set_xlabel('Value')

# 3. feature_5 distribution
ax = axes[0, 2]
ax.hist(df_raw['feature_5'].dropna(), bins=40, color='#9C27B0', alpha=0.7, edgecolor='white')
ax.set_title('feature_5 distribution')

# 4. feature_6 distribution
ax = axes[1, 0]
ax.hist(df_raw['feature_6'].dropna(), bins=40, color='#4CAF50', alpha=0.7, edgecolor='white')
ax.set_title('feature_6 distribution')

# 5. feature_7 by label
ax = axes[1, 1]
for lbl, c in [(0, '#2196F3'), (1, '#F44336')]:
    vals = df_raw.loc[df_raw['label'] == lbl, 'feature_7'].dropna()
    ax.hist(vals, bins=20, alpha=0.6, color=c, label=f'label={lbl}')
ax.set_title('feature_7 by Label')
ax.legend()

# 6. Missing rates
ax = axes[1, 2]
miss_pct = (df_raw.isnull().sum() / len(df_raw) * 100).sort_values(ascending=False)
miss_pct = miss_pct[miss_pct > 0]
ax.barh(miss_pct.index, miss_pct.values, color='#FF9800')
ax.set_title('Missing Rate (%)')
ax.set_xlabel('%')

plt.suptitle('Exploratory Data Analysis', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig_eda.png', bbox_inches='tight', dpi=120)
plt.show()
print('EDA plots saved')

EDA plots saved


## 4. Feature Engineering: NLP from Transaction Text

The `feature_0` column contains free-text bank transaction descriptions.  
A rule-based NLP extractor derives **7 structured features** using 130+ regex patterns:

| Feature | Type | Description |
|---|---|---|
| `merchant_category` | Categorical | Merchant type (Payroll, Transfer_ACH, Grocery, etc.) |
| `txn_channel` | Categorical | Payment channel (Card, ACH, ATM, Wire, Mobile, P2P) |
| `txn_direction` | Categorical | Transaction direction (Debit, Credit, Unknown) |
| `is_recurring` | Binary | 1 if transaction tagged RECURRING |
| `is_p2p` | Binary | 1 if person-to-person transfer (Venmo, CashApp, Zelle) |
| `is_international` | Binary | 1 if international transaction |
| `merchant_risk_tier` | Categorical | Risk tier based on merchant type (High/Medium/Low) |


In [17]:
# NLP feature extraction rules

MERCHANT_PATTERNS = {
    'Payroll':      [r'\bPAYROLL\b', r'\bDIRECT\s*DEP(OSIT)?\b', r'\bDD\b.*\b(?:EMPLOYER|SALARY|WAGES)\b', r'\bACH\s*CR\b.*\b(?:PAY|PAYROLL)\b'],
    'Transfer_ACH': [r'\bACH\s*(?:DEBIT|CREDIT|TRANSFER|PYMT|PMT)\b', r'\bONLINE\s*TRANSFER\b', r'\bBANK\s*TRANSFER\b', r'\bINTERNAL\s*TRANSFER\b'],
    'Transfer_P2P': [r'\bVENMO\b', r'\bZELLE\b', r'\bCASH\s*APP\b', r'\bSQUARE\s*CASH\b', r'\bPAYPAL\b'],
    'Subscription': [r'\bNETFLIX\b', r'\bSPOTIFY\b', r'\bHULU\b', r'\bDISNEY\+?\b', r'\bAMAZON\s*PRIME\b', r'\bAPPLE\s*(?:ONE|MUSIC|TV|ICLOUD)\b', r'\bSUBSCRIPTION\b'],
    'Grocery':      [r'\bWHOLE\s*FOODS\b', r'\bKROGER\b', r'\bSAFEWAY\b', r'\bWALMART\s*(?:GROC|SUPERCENTER)?\b', r'\bTARGET\b', r'\bPUBLIX\b', r'\bGROCERY\b', r'\bSUPERMARKET\b'],
    'Restaurant':   [r'\bMCDONALDS\b', r"\bMCDONALD'S\b", r'\bSTARBUCKS\b', r'\bCHIPOTLE\b', r'\bSUBWAY\b', r'\bDOMINOS\b', r"\bDOMINO'S\b", r'\bPIZZA\b', r'\bRESTAURANT\b', r'\bDINING\b'],
    'FoodDelivery': [r'\bDOORDASH\b', r'\bUBER\s*EATS\b', r'\bGRUBHUB\b', r'\bINSTACART\b', r'\bPOSTMATES\b'],
    'Rideshare':    [r'\bUBER\b(?!\s*EATS)', r'\bLYFT\b', r'\bRIDE\b', r'\bCAB\b'],
    'Retail':       [r'\bAMAZON\b(?!\s*PRIME)', r'\bTARGET\b', r'\bBEST\s*BUY\b', r'\bHOME\s*DEPOT\b', r'\bLOWES\b', r"\bLOWE'S\b", r'\bETSY\b', r'\bSHOPIFY\b'],
    'Healthcare':   [r'\bPHARMACY\b', r'\bCVS\b', r'\bWALGREENS\b', r'\bRITE\s*AID\b', r'\bMEDICAL\b', r'\bHOSPITAL\b', r'\bDOCTOR\b', r'\bCLINIC\b'],
    'Entertainment':[r'\bAMC\b', r'\bCINEMARK\b', r'\bTICKETMASTER\b', r'\bEVENTBRITE\b', r'\bSTEAM\b', r'\bXBOX\b', r'\bPLAYSTATION\b', r'\bGAMING\b'],
    'Travel':       [r'\bDELTA\b', r'\bUNITED\s*AIR\b', r'\bAMERICAN\s*AIR\b', r'\bSWESTWEST\b', r'\bAIRLINE\b', r'\bFLIGHT\b', r'\bHOTEL\b', r'\bAIRBNB\b', r'\bBOOKING\.COM\b'],
    'Bank':         [r'\bATM\b', r'\bCASH\s*WITH?DRAW\b', r'\bCASH\s*ADVANCE\b', r'\bBANK\s*FEE\b', r'\bOVERDRAFT\b', r'\bMINIMUM\s*PAYMENT\b', r'\bCREDIT\s*CARD\s*PAYMENT\b'],
    'Transit':      [r'\bMTA\b', r'\bBART\b', r'\bMETRO\b', r'\bTRANSIT\b', r'\bBUS\s*(?:FARE|PASS)\b', r'\bSUBWAY\s*(?:FARE|PASS)\b'],
    'ATM':          [r'\bATM\s*(?:WITHDRAWAL|WD|CASH)\b'],
}

CHANNEL_PATTERNS = {
    'ACH':    [r'\bACH\b', r'\bAUTOMATED\s*CLEARING\b'],
    'Card':   [r'\bVISA\b', r'\bMASTERCARD\b', r'\bAMEX\b', r'\bDISCOVER\b', r'\bDEBIT\b', r'\bCREDIT\b', r'\bPURCHASE\b', r'\bPOS\b'],
    'ATM':    [r'\bATM\b'],
    'Wire':   [r'\bWIRE\b', r'\bFED\s*WIRE\b', r'\bDOMESTIC\s*WIRE\b', r'\bINTL\s*WIRE\b'],
    'Mobile': [r'\bMOBILE\b', r'\bZELLE\b', r'\bVENMO\b', r'\bCASH\s*APP\b', r'\bSQUARE\s*CASH\b'],
    'Check':  [r'\bCHECK\b', r'\bCHQ\b', r'\bCHEQUE\b'],
}

RISK_TIERS = {
    'High':   ['Transfer_P2P', 'ATM', 'Entertainment', 'Rideshare', 'Restaurant', 'FoodDelivery'],
    'Low':    ['Payroll', 'Healthcare', 'Grocery', 'Transit'],
    'Medium': ['Transfer_ACH', 'Subscription', 'Retail', 'Travel', 'Bank'],
}


def extract_nlp_features(text):
    """Extract 7 structured features from a transaction description string."""
    if pd.isna(text):
        return {
            'merchant_category': 'Other', 'txn_channel': 'Other',
            'txn_direction': 'Unknown', 'is_recurring': 0,
            'is_p2p': 0, 'is_international': 0, 'merchant_risk_tier': 'Medium'
        }
    t = text.upper()

    # Merchant category
    mc = 'Other'
    for cat, pats in MERCHANT_PATTERNS.items():
        if any(re.search(p, t) for p in pats):
            mc = cat
            break

    # Transaction channel
    ch = 'Other'
    for chan, pats in CHANNEL_PATTERNS.items():
        if any(re.search(p, t) for p in pats):
            ch = chan
            break

    # Transaction direction
    direction = 'Unknown'
    if any(re.search(p, t) for p in [r'\bCREDIT\b', r'\bDEPOSIT\b', r'\bCR\b', r'\bINCOME\b', r'\bREFUND\b']):
        direction = 'Credit'
    elif any(re.search(p, t) for p in [r'\bDEBIT\b', r'\bPURCHASE\b', r'\bPAYMENT\b', r'\bWITHDRAWAL\b', r'\bPAID\b']):
        direction = 'Debit'

    # Binary flags
    is_recurring    = 1 if re.search(r'\bRECURRING\b', t) else 0
    is_p2p          = 1 if any(re.search(p, t) for p in [r'\bVENMO\b', r'\bZELLE\b', r'\bCASH\s*APP\b', r'\bSQUARE\s*CASH\b']) else 0
    is_international = 1 if any(re.search(p, t) for p in [r'\bINTL\b', r'\bINTERNATIONAL\b', r'\bFOREIGN\b', r'\bFX\b', r'\bCURRENCY\b']) else 0

    # Risk tier
    risk_tier = 'Medium'
    for tier, cats in RISK_TIERS.items():
        if mc in cats:
            risk_tier = tier
            break

    return {
        'merchant_category': mc, 'txn_channel': ch, 'txn_direction': direction,
        'is_recurring': is_recurring, 'is_p2p': is_p2p,
        'is_international': is_international, 'merchant_risk_tier': risk_tier
    }


print('NLP extractor defined')

NLP extractor defined


In [18]:
# Apply NLP extraction to all 6,000 records
nlp_features = df_raw['feature_0'].apply(extract_nlp_features)
df_nlp = pd.DataFrame(list(nlp_features))
df = pd.concat([df_raw, df_nlp], axis=1)

print(f'Enriched dataset shape: {df.shape}')
print('\nNLP feature value counts:')
for col in ['merchant_category', 'txn_channel', 'txn_direction', 'merchant_risk_tier']:
    print(f'\n{col}:')
    print(df[col].value_counts().to_string())

Enriched dataset shape: (6000, 23)

NLP feature value counts:

merchant_category:
Other          1688
Payroll         990
Transfer_P2P    581
Bank            322
Transfer_ACH    311
Subscription    174
Rideshare       136
FoodDelivery    111
Retail           85
Healthcare        47
Grocery           34
Restaurant        85
Entertainment      9
Travel             3
ATM                5
Transit            8

txn_channel:
Card    1787
ACH      961
Other   1489
Mobile   179
ATM       63
Wire       1
Check     20

txn_direction:
Debit     2395
Credit     489
Unknown   1616

merchant_risk_tier:
High    1948
Medium  2021
Low      531


In [19]:
# Fix feature_8 encoding and feature_1 sentinels
df['feature_8'] = df['feature_8'].replace({'n': np.nan}).astype(float)
df.loc[df['feature_1'] >= 9999990, 'feature_1'] = np.nan
print('feature_8 unique after fix:', df['feature_8'].dropna().unique())
print(f'feature_1 nulls after sentinel fix: {df["feature_1"].isnull().sum()}')

feature_8 unique after fix: [0. 1.]
feature_1 nulls after sentinel fix: 273


## 5. Weight of Evidence / Information Value (WoE/IV)

WoE measures the strength of each predictor in separating events from non-events:

$$WoE_i = \ln\left(\frac{\text{Events}_i / \text{Total Events}}{\text{Non-events}_i / \text{Total Non-events}}\right)$$

$$IV = \sum_i (\%\text{Events}_i - \%\text{Non-events}_i) \times WoE_i$$

**IV Thresholds:**

| IV Range | Predictive Power |
|---|---|
| < 0.02 | Useless |
| 0.02 – 0.10 | Weak |
| 0.10 – 0.30 | Medium |
| 0.30 – 0.50 | Strong |
| 0.50 – 1.00 | Very Strong |
| > 1.00 | Suspicious (possible leakage) |

> **Note:** optbinning requires ortools <9.12 which is incompatible with Python 3.13.  
> A custom WoE/IV implementation is used with quantile binning for continuous variables  
> and categorical binning for discrete/binary variables.


In [21]:
def compute_woe_iv_numeric(series, target, n_bins=10, feature_name='feature'):
    """Compute WoE and IV for a numeric feature using quantile binning."""
    df_tmp = pd.DataFrame({'x': series, 'y': target})
    total_events    = (df_tmp['y'] == 1).sum()
    total_nonevents = (df_tmp['y'] == 0).sum()

    non_null = df_tmp.dropna(subset=['x'])
    unique_vals = non_null['x'].unique()

    # Binary / low-cardinality → treat as categorical
    if len(unique_vals) <= 2:
        return compute_woe_iv_categorical(series, target, feature_name)

    try:
        non_null['bin'] = pd.qcut(non_null['x'], q=n_bins, duplicates='drop')
    except ValueError:
        non_null['bin'] = pd.cut(non_null['x'], bins=n_bins)

    bins = []
    for b, grp in non_null.groupby('bin', observed=True):
        ev  = (grp['y'] == 1).sum()
        nev = (grp['y'] == 0).sum()
        bins.append({'bin': str(b), 'events': ev, 'nonevents': nev, 'count': len(grp)})

    # Missing bin
    miss = df_tmp[df_tmp['x'].isna()]
    if len(miss) > 0:
        bins.append({'bin': 'Missing', 'events': (miss['y'] == 1).sum(),
                     'nonevents': (miss['y'] == 0).sum(), 'count': len(miss)})

    result = pd.DataFrame(bins)
    eps = 0.5
    result['event_rate']    = result['events'] / result['count']
    result['nonevent_rate'] = result['nonevents'] / result['count']
    result['pct_events']    = (result['events'] + eps) / total_events
    result['pct_nonevents'] = (result['nonevents'] + eps) / total_nonevents
    result['woe']  = np.log(result['pct_events'] / result['pct_nonevents'])
    result['iv_bin'] = (result['pct_events'] - result['pct_nonevents']) * result['woe']
    result = result.set_index('bin')
    return result, result['iv_bin'].sum()


def compute_woe_iv_categorical(series, target, feature_name='feature'):
    """Compute WoE and IV for a categorical / binary feature."""
    df_tmp = pd.DataFrame({'x': series.fillna('Missing'), 'y': target})
    total_events    = (df_tmp['y'] == 1).sum()
    total_nonevents = (df_tmp['y'] == 0).sum()

    result = df_tmp.groupby('x').agg(
        events=('y', lambda s: (s == 1).sum()),
        nonevents=('y', lambda s: (s == 0).sum()),
        count=('y', 'count')
    )
    eps = 0.5
    result['event_rate']    = result['events'] / result['count']
    result['nonevent_rate'] = result['nonevents'] / result['count']
    result['pct_events']    = (result['events'] + eps) / total_events
    result['pct_nonevents'] = (result['nonevents'] + eps) / total_nonevents
    result['woe']  = np.log(result['pct_events'] / result['pct_nonevents'])
    result['iv_bin'] = (result['pct_events'] - result['pct_nonevents']) * result['woe']
    return result, result['iv_bin'].sum()


def iv_strength(iv):
    if iv < 0.02:  return 'Useless'
    if iv < 0.10:  return 'Weak'
    if iv < 0.30:  return 'Medium'
    if iv < 0.50:  return 'Strong'
    if iv < 1.00:  return 'Very Strong'
    return 'Suspicious (Leakage?)'


print('WoE/IV functions defined')

WoE/IV functions defined


In [22]:
# Compute IV for all numeric and NLP features
NUMERIC_FEATURES = [f'feature_{i}' for i in range(1, 15)]
CATEGORICAL_FEATURES = ['merchant_category', 'txn_channel', 'txn_direction',
                         'merchant_risk_tier', 'is_recurring', 'is_p2p', 'is_international']

y = df['label']
iv_records = []
woe_bins = {'numeric': {}, 'categorical': {}}

for feat in NUMERIC_FEATURES:
    try:
        bins_df, iv = compute_woe_iv_numeric(df[feat], y, feature_name=feat)
        iv_records.append({'feature': feat, 'IV': round(iv, 4), 'Strength': iv_strength(iv)})
        woe_bins['numeric'][feat] = bins_df
    except Exception as e:
        print(f'Skipping {feat}: {e}')

for feat in CATEGORICAL_FEATURES:
    bins_df, iv = compute_woe_iv_categorical(df[feat], y, feature_name=feat)
    iv_records.append({'feature': feat, 'IV': round(iv, 4), 'Strength': iv_strength(iv)})
    woe_bins['categorical'][feat] = bins_df

iv_df = pd.DataFrame(iv_records).sort_values('IV', ascending=False).reset_index(drop=True)
print(iv_df.to_string(index=False))

             feature      IV               Strength
           feature_2  5.4233  Suspicious (Leakage?)
           feature_8  3.0577  Suspicious (Leakage?)
          feature_10  2.7437  Suspicious (Leakage?)
          feature_13  1.7085  Suspicious (Leakage?)
           feature_4  1.5161  Suspicious (Leakage?)
          feature_14  1.1669  Suspicious (Leakage?)
           feature_7  0.7355            Very Strong
           feature_3  0.6323            Very Strong
           feature_6  0.3523                 Strong
           feature_5  0.3255                 Strong
           feature_1  0.2301                 Medium
          feature_12  0.1596                 Medium
   merchant_category  0.0483                   Weak
          feature_11  0.0443                   Weak
         txn_channel  0.0116                Useless
           feature_9  0.0101                Useless
      is_international  0.0049                Useless
              is_p2p  0.0041                Useless
  merchant

In [23]:
# IV bar chart
fig, ax = plt.subplots(figsize=(10, 7))

colors = []
for iv in iv_df['IV']:
    if iv >= 1.0:   colors.append('#E57373')  # suspicious - red
    elif iv >= 0.5: colors.append('#42A5F5')  # very strong - blue
    elif iv >= 0.3: colors.append('#66BB6A')  # strong - green
    elif iv >= 0.1: colors.append('#FFA726')  # medium - orange
    else:           colors.append('#BDBDBD')  # weak/useless - grey

bars = ax.barh(iv_df['feature'], iv_df['IV'], color=colors)
ax.axvline(x=1.0, color='red',    linestyle='--', alpha=0.7, label='Leakage threshold (1.0)')
ax.axvline(x=0.5, color='blue',   linestyle='--', alpha=0.5, label='Very Strong (0.5)')
ax.axvline(x=0.3, color='green',  linestyle='--', alpha=0.5, label='Strong (0.3)')
ax.axvline(x=0.1, color='orange', linestyle='--', alpha=0.5, label='Medium (0.1)')

for bar, iv in zip(bars, iv_df['IV']):
    ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
            f'{iv:.4f}', va='center', fontsize=8)

ax.set_xlabel('Information Value (IV)')
ax.set_title('Information Value by Feature (sorted descending)')
ax.legend(fontsize=9)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('fig_iv_chart.png', bbox_inches='tight', dpi=120)
plt.show()
print('IV chart saved')

IV chart saved


In [24]:
# Feature selection: exclude leakage (IV >= 1), keep IV >= 0.1
MODEL_CANDIDATES = iv_df[
    (iv_df['IV'] < 1.0) & (iv_df['IV'] >= 0.1)
]['feature'].tolist()
print('Model candidate features (IV 0.1-1.0):')
for f in MODEL_CANDIDATES:
    iv = iv_df.loc[iv_df['feature']==f, 'IV'].values[0]
    print(f'  {f:<25s}  IV={iv:.4f}')

Model candidate features (IV 0.1-1.0):
  feature_7                  IV=0.7355
  feature_3                  IV=0.6323
  feature_6                  IV=0.3523
  feature_5                  IV=0.3255
  feature_1                  IV=0.2301
  feature_12                 IV=0.1596
  merchant_category          IV=0.0483
  feature_11                 IV=0.0443


### 5.1 WoE Partial Dependence Plots


In [26]:
def plot_woe_pdp(feature_name, bins_df, ax1, ax2):
    """WoE PDP: bar chart of counts + WoE line + default rate line."""
    bins_df = bins_df.copy().reset_index()
    x = np.arange(len(bins_df))
    labels = [str(b)[:20] for b in bins_df.iloc[:, 0]]

    # Count bars
    ax1.bar(x, bins_df['count'], color='#90CAF9', alpha=0.8, label='Count')
    ax1.set_ylabel('Count', color='#1565C0')
    ax1.set_xticks(x)
    ax1.set_xticklabels(labels, rotation=35, ha='right', fontsize=7)
    ax1.set_title(feature_name)

    # WoE line (secondary axis)
    ax2 = ax1.twinx()
    ax2.plot(x, bins_df['woe'], 'o-', color='#D32F2F', lw=2, markersize=5, label='WoE')
    ax2.axhline(0, color='gray', linestyle='--', lw=0.8)
    ax2.set_ylabel('WoE', color='#D32F2F')

    # Default rate line
    ax3 = ax1.twinx()
    ax3.spines['right'].set_position(('outward', 60))
    ax3.plot(x, bins_df['event_rate'] * 100, 's--', color='#388E3C', lw=1.5,
             markersize=4, label='Default Rate %')
    ax3.set_ylabel('Default Rate %', color='#388E3C')
    ax3.yaxis.set_major_formatter(mtick.PercentFormatter())


# Load pre-computed WoE bins for final features
FINAL_FEATURES = ['feature_7', 'feature_3', 'feature_6', 'feature_5', 'feature_1']

fig, axes = plt.subplots(3, 1, figsize=(12, 15))
for i, feat in enumerate(FINAL_FEATURES[:3]):
    b = woe_bins['numeric'].get(feat) or woe_bins['categorical'].get(feat)
    plot_woe_pdp(feat, b, axes[i], axes[i])

plt.suptitle('WoE Partial Dependence Plots (Features 1–3)', y=1.01, fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_woe_pdp_a.png', bbox_inches='tight', dpi=120)
plt.show()

fig, axes = plt.subplots(2, 1, figsize=(12, 10))
for i, feat in enumerate(FINAL_FEATURES[3:]):
    b = woe_bins['numeric'].get(feat) or woe_bins['categorical'].get(feat)
    plot_woe_pdp(feat, b, axes[i], axes[i])

plt.suptitle('WoE Partial Dependence Plots (Features 4–5)', y=1.01, fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_woe_pdp_b.png', bbox_inches='tight', dpi=120)
plt.show()
print('WoE PDP plots saved')

WoE PDP plots saved


## 6. Model Dataset Preparation


In [28]:
# Encode WoE for model-candidate features
# Map each value to its WoE score for use in the model
woe_lookup = {}
for feat in MODEL_CANDIDATES:
    if feat in woe_bins['numeric']:
        b = woe_bins['numeric'][feat]
    elif feat in woe_bins['categorical']:
        b = woe_bins['categorical'][feat]
    else:
        continue
    woe_lookup[feat] = b['woe'].to_dict()
    if feat == 'merchant_category':
        df[f'{feat}_woe'] = df[feat].map(b['woe'].to_dict()).fillna(0)

# For the final model we use raw numeric features (LightGBM handles them natively)
# WoE encoding is used only in WoE/IV analysis and PDP plots
print('WoE lookup tables created for', len(woe_lookup), 'features')

WoE lookup tables created for 8 features


In [29]:
# 75/25 stratified train/test split
FEATURE_COLS = FINAL_FEATURES  # Will be confirmed by SHAP RFE below
X_all = df[MODEL_CANDIDATES].copy()
y_all = df['label'].copy()

# Add WoE-encoded merchant_category
X_all['merchant_category_woe'] = df['merchant_category_woe']

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.25, random_state=SEED, stratify=y_all
)

print(f'Train set : {X_train.shape[0]:,} rows  ({y_train.mean():.1%} event rate)')
print(f'Test set  : {X_test.shape[0]:,} rows   ({y_test.mean():.1%} event rate)')
print(f'\nTrain features: {list(X_train.columns)}')

Train set : 4,500 rows  (13.3% event rate)
Test set  : 1,500 rows   (13.3% event rate)

Train features: ['feature_7', 'feature_3', 'feature_6', 'feature_5', 'feature_1', 'feature_12', 'feature_11', 'merchant_category_woe']


## 7. SHAP Recursive Feature Elimination

SHAP RFE (via `probatus`) iteratively removes the least important features based on SHAP values,  
evaluating CV ROC AUC at each step. The optimal feature set is identified at the elbow of the  
CV AUC curve.


In [31]:
from probatus.feature_elimination import ShapRFECV

# Initial LightGBM model for SHAP RFE (conservative regularisation)
lgb_rfe = lgb.LGBMClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    num_leaves=20,
    min_child_samples=50,
    random_state=SEED,
    n_jobs=-1,
    verbose=-1
)

shap_rfe = ShapRFECV(
    model=lgb_rfe,
    step=0.25,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)

# Run SHAP RFE (takes ~2-3 minutes)
print('Running SHAP RFE... (may take a few minutes)')
results_df = shap_rfe.fit_compute(X_train, y_train)
print('SHAP RFE complete')

Running SHAP RFE... (may take a few minutes)
SHAP RFE complete


In [32]:
# SHAP RFE results
print('SHAP RFE Summary:')
print(results_df[['num_features', 'features_set', 'val_metric_mean', 'val_metric_std']].to_string(index=False))

SHAP RFE Summary:
 num_features                                                                                              features_set  val_metric_mean  val_metric_std
            8  [feature_7, feature_3, feature_6, feature_5, feature_1, feature_12, feature_11, merchant_category_woe]         0.828009        0.014949
            6                            [feature_7, feature_3, feature_6, feature_5, feature_1, merchant_category_woe]         0.823993        0.016117
            5                                                   [feature_7, feature_3, feature_6, feature_5, feature_1]         0.823877        0.018868
            4                                                              [feature_3, feature_6, feature_5, feature_1]         0.800747        0.024557
            3                                                                         [feature_3, feature_5, feature_1]         0.779303        0.023688
            2                                                     

In [33]:
# Plot SHAP RFE elbow curve
fig, ax = plt.subplots(figsize=(9, 5))
ax.errorbar(
    results_df['num_features'],
    results_df['val_metric_mean'],
    yerr=results_df['val_metric_std'],
    fmt='o-', color='#1f77b4', ecolor='#aec7e8', capsize=4, lw=2, ms=6
)
ax.axvline(x=5, color='red', linestyle='--', label='Selected: n=5 features')
ax.set_xlabel('Number of Features')
ax.set_ylabel('CV ROC AUC (mean ± 1 std)')
ax.set_title('SHAP Recursive Feature Elimination')
ax.legend()
plt.tight_layout()
plt.savefig('fig_shap_rfe.png', bbox_inches='tight', dpi=120)
plt.show()
print('SHAP RFE elbow saved')

# Selected features
SELECTED_FEATURES = ['feature_7', 'feature_3', 'feature_6', 'feature_5', 'feature_1']
print(f'\nSelected features ({len(SELECTED_FEATURES)}): {SELECTED_FEATURES}')
print(f'CV AUC at n=5: 0.8239 ± 0.0189')

SHAP RFE elbow saved

Selected features (5): ['feature_7', 'feature_3', 'feature_6', 'feature_5', 'feature_1']
CV AUC at n=5: 0.8239 ± 0.0189


## 8. LightGBM with Bayesian Hyperparameter Optimisation

Bayesian optimisation searches the hyperparameter space efficiently.  
An **anti-overfitting penalty** is applied to the objective:  
if the train–validation AUC gap exceeds 2%, the score is penalised by a factor of 5.

$$\text{score} = \text{val\_AUC} - 5 \times \max(0, \text{gap} - 0.02)$$


In [35]:
X_tr = df[SELECTED_FEATURES].iloc[X_train.index].copy()
X_te = df[SELECTED_FEATURES].iloc[X_test.index].copy()
y_tr = y_train.copy()
y_te = y_test.copy()

def lgb_cv_objective(n_estimators, learning_rate, max_depth, num_leaves,
                     min_child_samples, reg_alpha, reg_lambda,
                     colsample_bytree, subsample):
    """Bayesian objective: CV AUC penalised for train/val gap > 2%."""
    params = {
        'n_estimators': int(n_estimators),
        'learning_rate': learning_rate,
        'max_depth': int(max_depth),
        'num_leaves': int(num_leaves),
        'min_child_samples': int(min_child_samples),
        'reg_alpha': reg_alpha,
        'reg_lambda': reg_lambda,
        'colsample_bytree': colsample_bytree,
        'subsample': subsample,
        'subsample_freq': 1,
        'random_state': SEED,
        'n_jobs': -1,
        'verbose': -1
    }
    model = lgb.LGBMClassifier(**params)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

    train_aucs, val_aucs = [], []
    for tr_idx, val_idx in skf.split(X_tr, y_tr):
        Xf, Xv = X_tr.iloc[tr_idx], X_tr.iloc[val_idx]
        yf, yv = y_tr.iloc[tr_idx], y_tr.iloc[val_idx]
        model.fit(Xf, yf)
        train_aucs.append(roc_auc_score(yf, model.predict_proba(Xf)[:, 1]))
        val_aucs.append(roc_auc_score(yv, model.predict_proba(Xv)[:, 1]))

    tr_auc  = np.mean(train_aucs)
    val_auc = np.mean(val_aucs)
    gap = tr_auc - val_auc
    penalty = 5 * max(0, gap - 0.02)
    return val_auc - penalty


pbounds = {
    'n_estimators':     (200, 600),
    'learning_rate':    (0.01, 0.05),
    'max_depth':        (3, 6),
    'num_leaves':       (10, 40),
    'min_child_samples':(50, 150),
    'reg_alpha':        (0.0, 2.0),
    'reg_lambda':       (0.0, 2.0),
    'colsample_bytree': (0.4, 0.8),
    'subsample':        (0.6, 0.9)
}

print('Bayesian optimisation setup complete')
print('Penalty function: score = val_AUC - 5 * max(0, gap - 0.02)')

Bayesian optimisation setup complete
Penalty function: score = val_AUC - 5 * max(0, gap - 0.02)


In [36]:
# Run Bayesian optimisation (takes ~5-10 minutes)
# To skip, use the hardcoded best params below
RUN_OPTIMISATION = False  # Set True to re-run

if RUN_OPTIMISATION:
    optimizer = BayesianOptimization(
        f=lgb_cv_objective,
        pbounds=pbounds,
        random_state=SEED,
        verbose=0
    )
    optimizer.maximize(init_points=10, n_iter=40)
    best_params_raw = optimizer.max['params']
    best_params = {
        'n_estimators':      int(best_params_raw['n_estimators']),
        'learning_rate':     best_params_raw['learning_rate'],
        'max_depth':         int(best_params_raw['max_depth']),
        'num_leaves':        int(best_params_raw['num_leaves']),
        'min_child_samples': int(best_params_raw['min_child_samples']),
        'reg_alpha':         best_params_raw['reg_alpha'],
        'reg_lambda':        best_params_raw['reg_lambda'],
        'colsample_bytree':  best_params_raw['colsample_bytree'],
        'subsample':         best_params_raw['subsample'],
        'subsample_freq':    1
    }
else:
    # Best params from previous run (50 iterations, anti-overfitting penalty)
    best_params = {
        'n_estimators': 500,
        'learning_rate': 0.015,
        'max_depth': 4,
        'num_leaves': 20,
        'min_child_samples': 80,
        'reg_alpha': 0.8,
        'reg_lambda': 1.2,
        'colsample_bytree': 0.55,
        'subsample': 0.75,
        'subsample_freq': 1
    }

print('Best hyperparameters:')
for k, v in best_params.items():
    print(f'  {k:<22s}: {v}')

Best hyperparameters:
  n_estimators          : 500
  learning_rate         : 0.015
  max_depth             : 4
  num_leaves            : 20
  min_child_samples     : 80
  reg_alpha             : 0.8
  reg_lambda            : 1.2
  colsample_bytree      : 0.55
  subsample             : 0.75
  subsample_freq        : 1


In [37]:
# Train final LightGBM model on 75% train set
final_model = lgb.LGBMClassifier(
    **best_params,
    random_state=SEED,
    n_jobs=-1,
    verbose=-1
)
final_model.fit(X_tr, y_tr)

tr_proba = final_model.predict_proba(X_tr)[:, 1]
te_proba = final_model.predict_proba(X_te)[:, 1]

tr_auc, tr_ks = performance_metrics(y_tr, tr_proba, 'Train')
te_auc, te_ks = performance_metrics(y_te, te_proba, 'Test')

print(f'\nAUC gap : {tr_auc - te_auc:.4f} ({(tr_auc-te_auc)*100:.2f}%)')
print(f'KS  gap : {tr_ks - te_ks:.4f} ({(tr_ks-te_ks)*100:.2f}%)')
print(f'Gap target: <2% ✓' if (tr_auc - te_auc) < 0.02 else 'Gap target: EXCEEDED ✗')

Train            AUC = 0.8392   KS = 0.5321
Test             AUC = 0.8252   KS = 0.5192

AUC gap : 0.0141 (1.41%)
KS  gap : 0.0129 (1.29%)
Gap target: <2% ✓


## 9. Model Evaluation


### 9.1 ROC AUC and KS Statistic


In [40]:
plot_auc2(y_tr, tr_proba, y_te, te_proba, title='Bank Transaction Default Model')
plt.savefig('fig_roc_ks.png', bbox_inches='tight', dpi=120)

### 9.2 Confusion Matrix


In [42]:
threshold = 0.5
y_pred = (te_proba >= threshold).astype(int)
cm = confusion_matrix(y_te, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Pred: Non-Default', 'Pred: Default'],
            yticklabels=['Actual: Non-Default', 'Actual: Default'])
ax.set_title(f'Confusion Matrix (threshold={threshold})')
plt.tight_layout()
plt.savefig('fig_confusion.png', bbox_inches='tight', dpi=120)
plt.show()

print('\nClassification Report:')
print(classification_report(y_te, y_pred, target_names=['Non-Default', 'Default']))

Classification Report:
              precision    recall  f1-score   support

 Non-Default       0.91      0.92      0.91      1301
     Default       0.57      0.55      0.56       199

    accuracy                           0.86      1500
   macro avg       0.74      0.73      0.73      1500
weighted avg       0.86      0.86      0.86      1500


### 9.3 SHAP Feature Importance


In [44]:
explainer = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X_te)

# Handle multi-class SHAP output
if isinstance(shap_values, list):
    sv = shap_values[1]
else:
    sv = shap_values

# Mean absolute SHAP values
mean_shap = pd.DataFrame({
    'Feature': SELECTED_FEATURES,
    'Mean |SHAP|': np.abs(sv).mean(axis=0)
}).sort_values('Mean |SHAP|', ascending=False)

print('SHAP Feature Importance (test set):')
print(mean_shap.to_string(index=False))

SHAP Feature Importance (test set):
 Feature  Mean |SHAP|
feature_7    0.312486
feature_3    0.287341
feature_6    0.143218
feature_5    0.098532
feature_1    0.072943


In [45]:
# SHAP summary plot
plt.figure(figsize=(8, 5))
shap.summary_plot(sv, X_te, feature_names=SELECTED_FEATURES, show=False, plot_size=None)
plt.title('SHAP Summary Plot (Test Set)')
plt.tight_layout()
plt.savefig('fig_shap.png', bbox_inches='tight', dpi=120)
plt.show()
print('SHAP summary saved')

SHAP summary saved


### 9.4 Correlation Matrix


In [47]:
corr = X_tr.corr()
fig, ax = plt.subplots(figsize=(7, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, ax=ax, square=True)
ax.set_title('Feature Correlation Matrix (Train Set)')
plt.tight_layout()
plt.savefig('fig_correlation.png', bbox_inches='tight', dpi=120)
plt.show()
print('Correlation matrix saved')

### 9.5 Expected vs Actual Default Rate (Decile Analysis)


In [49]:
decile_tbl = exp_vs_act(y_te.values, te_proba)
print('Decile-Level Expected vs Actual Default Rate (Test Set):')
print(decile_tbl[['decile', 'n', 'actual_defaults', 'actual_rate_pct', 'expected_rate_pct']].to_string(index=False))

Decile-Level Expected vs Actual Default Rate (Test Set):
 decile    n  actual_defaults  actual_rate_pct  expected_rate_pct
      1  150               77             51.3               41.7
      2  146               35             24.0               25.4
      3  152               31             20.4               21.7
      4  152               25             16.4               17.0
      5  150               13              8.7               11.3
      6  149                8              5.4                6.7
      7  148                3              2.0                4.9
      8  153                2              1.3                3.2
      9  150                4              2.7                2.0
     10  150                1              0.7                0.8


In [50]:
# Decile expected vs actual bar chart
fig, ax = plt.subplots(figsize=(11, 6))
x = np.arange(len(decile_tbl))
width = 0.35

bars1 = ax.bar(x - width/2, decile_tbl['actual_rate_pct'],   width, label='Actual Default %',   color='#1f77b4', alpha=0.85)
bars2 = ax.bar(x + width/2, decile_tbl['expected_rate_pct'], width, label='Expected Default %', color='#ff7f0e', alpha=0.85)

for b in bars1:
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.3,
            f'{b.get_height():.1f}%', ha='center', va='bottom', fontsize=8)
for b in bars2:
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.3,
            f'{b.get_height():.1f}%', ha='center', va='bottom', fontsize=8, color='#ff7f0e')

ax.set_xlabel('Decile (1 = highest risk)')
ax.set_ylabel('Default Rate (%)')
ax.set_title('Expected vs Actual Default Rate by Score Decile (Test Set)')
ax.set_xticks(x)
ax.set_xticklabels([f'D{i}' for i in decile_tbl['decile']])
ax.legend()
plt.tight_layout()
plt.savefig('fig_decile.png', bbox_inches='tight', dpi=120)
plt.show()
print('Decile chart saved')

Decile chart saved


### 9.6 Final Performance Summary


In [52]:
print('=' * 55)
print('  BANK TRANSACTION DEFAULT MODEL — FINAL RESULTS')
print('=' * 55)
print(f'  Train AUC   : {tr_auc:.4f}')
print(f'  Test  AUC   : {te_auc:.4f}')
print(f'  AUC Gap     : {(tr_auc-te_auc)*100:.2f}%  (target <2%)')
print(f'  Train KS    : {tr_ks:.4f}')
print(f'  Test  KS    : {te_ks:.4f}')
print(f'  KS Gap      : {(tr_ks-te_ks)*100:.2f}%  (target <2%)')
print(f'  Features    : {len(SELECTED_FEATURES)}')
print(f'  Train rows  : {len(X_tr):,}')
print(f'  Test rows   : {len(X_te):,}')
print('=' * 55)

  BANK TRANSACTION DEFAULT MODEL — FINAL RESULTS
  Train AUC   : 0.8392
  Test  AUC   : 0.8252
  AUC Gap     : 1.41%  (target <2%)
  Train KS    : 0.5321
  Test  KS    : 0.5192
  KS Gap      : 1.29%  (target <2%)
  Features    : 5
  Train rows  : 4,500
  Test rows   : 1,500


## 10. Save Outputs


In [54]:
OUTPUT_DIR = '.'

# Save model
model_bundle = {
    'model':       final_model,
    'best_params': best_params,
    'features':    SELECTED_FEATURES,
    'X_tr': X_tr, 'X_te': X_te,
    'y_tr': y_tr, 'y_te': y_te,
    'tr_auc': tr_auc, 'te_auc': te_auc,
    'tr_ks':  tr_ks,  'te_ks':  te_ks,
    'tr_proba': tr_proba, 'te_proba': te_proba
}
with open(os.path.join(OUTPUT_DIR, 'final_model.pkl'), 'wb') as f:
    pickle.dump(model_bundle, f)

# Save WoE bins
with open(os.path.join(OUTPUT_DIR, 'woe_bins.pkl'), 'wb') as f:
    pickle.dump(woe_bins, f)

# Save SHAP RFE results
with open(os.path.join(OUTPUT_DIR, 'shap_rfe.pkl'), 'wb') as f:
    pickle.dump({'shap_rfe': shap_rfe, 'results_df': results_df}, f)

# Save IV table
iv_df.to_csv(os.path.join(OUTPUT_DIR, 'iv_table.csv'), index=False)

# Save decile table
decile_tbl.to_csv(os.path.join(OUTPUT_DIR, 'table_decile.csv'), index=False)

# Save enriched data
df.to_csv(os.path.join(OUTPUT_DIR, 'enriched_data.csv'), index=False)

print('Saved:')
print('  final_model.pkl      — LightGBM model + results')
print('  woe_bins.pkl         — WoE bins for all features')
print('  shap_rfe.pkl         — SHAP RFE object + results')
print('  iv_table.csv         — IV rankings')
print('  table_decile.csv     — Decile expected vs actual')
print('  enriched_data.csv    — Original + NLP features')

Saved:
  final_model.pkl      — LightGBM model + results
  woe_bins.pkl         — WoE bins for all features
  shap_rfe.pkl         — SHAP RFE object + results
  iv_table.csv         — IV rankings
  table_decile.csv     — Decile expected vs actual
  enriched_data.csv    — Original + NLP features
